# Lab 00 · Arrume a casa antes de modelar
**Machine Learning na Prática · Dia 1 · Módulo 01: Dados & Datasets**

Neste notebook você vai fazer o que todo projeto de ML faz **antes** de treinar qualquer modelo.
Aqui **não há modelo nem métricas**: o objetivo é deixar os dados prontos. O treino começa no Lab 01, já com estes dados limpos.

| Etapa | O que fazer |
|---|---|
| 1. Inspecionar | formato, tipos, estatísticas, faltantes |
| 2. Limpar | duplicados, categorias, outliers, alvo faltante |
| 3. Explorar | histogramas, boxplot, correlação |
| 4. Preparar | features × alvo, split, one-hot, padronização e `Pipeline` sem vazamento |

Procure as células marcadas com **`# TODO`** e substitua os `___` pelo código certo.
Rode as células em ordem (`Shift + Enter`).

### Cola rápida de pandas
| Para… | use |
|---|---|
| ler um CSV | `pd.read_csv('arquivo.csv')` |
| formato (linhas, colunas) / tipos | `df.shape` / `df.info()` |
| estatísticas das colunas numéricas | `df.describe()` |
| faltantes por coluna | `df.isna().sum()` |
| contar valores de uma coluna | `df['col'].value_counts()` |
| achar / remover duplicados | `df.duplicated().sum()` / `df.drop_duplicates()` |
| limpar texto | `df['col'].str.strip().str.lower()` |
| remover linhas com faltante numa coluna | `df.dropna(subset=['col'])` |
| histogramas | `df.hist()` |
| correlação | `df.corr(numeric_only=True)` |
| média de um grupo | `df.groupby('col')['outra'].mean()` |

### Bibliotecas
No **Google Colab** tudo já vem instalado: esta célula só confere. Em outro ambiente (Jupyter no seu PC), ela instala o que faltar.

In [ ]:
import importlib.util, subprocess, sys
pacotes = {'numpy': 'numpy', 'pandas': 'pandas', 'matplotlib': 'matplotlib', 'sklearn': 'scikit-learn'}
faltando = [pip for modulo, pip in pacotes.items() if importlib.util.find_spec(modulo) is None]
if faltando:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *faltando])
print('Bibliotecas prontas!' if not faltando else f'Instalado agora: {", ".join(faltando)}')

## 0 · Preparação

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

pd.set_option('display.max_columns', 20)
plt.rcParams['figure.figsize'] = (8, 4)
print('Tudo pronto!')

### O dataset: desempenho de estudantes
Os dados abaixo são **sintéticos** (gerados aqui mesmo, para não depender de download), mas imitam problemas reais:
alguém digitou errado, algumas linhas foram copiadas duas vezes, o turno foi escrito de jeitos diferentes…

Rode a célula para criar o arquivo `alunos.csv`. Você **não precisa** entender o código de geração agora.

In [ ]:
def gerar_dados(n=400, seed=42):
    rng = np.random.default_rng(seed)
    horas = np.clip(rng.normal(6, 2.2, n), 0.5, 14).round(1)
    faltas = rng.poisson(3, n)
    turno = rng.choice(['manhã', 'tarde', 'noite'], n, p=[.45, .3, .25])
    monitoria = rng.choice(['sim', 'não'], n, p=[.35, .65])
    idade = rng.integers(17, 31, n)
    renda = np.round(rng.lognormal(8.2, 0.5, n), -1)          # R$ por mês, escala grande
    nota = (2.0 + 0.45 * horas - 0.25 * faltas + 0.6 * (monitoria == 'sim')
            - 0.4 * (turno == 'noite') + rng.normal(0, 0.9, n))
    nota = np.clip(nota, 0, 10).round(1)
    df = pd.DataFrame({'horas_estudo': horas, 'faltas': faltas, 'turno': turno,
                       'monitoria': monitoria, 'idade': idade, 'renda_familiar': renda,
                       'nota_final': nota})
    df['situacao'] = np.where(df['nota_final'] >= 6, 'aprovado', 'reprovado')

    # ---- problemas do mundo real ----
    sujo = rng.choice(n, 40, replace=False)
    df.loc[sujo[:14], 'turno'] = rng.choice(['Manhã', 'MANHÃ ', 'manha', 'Noite', ' tarde'], 14)
    df.loc[rng.choice(n, 20, replace=False), 'horas_estudo'] = np.nan
    df.loc[rng.choice(n, 15, replace=False), 'faltas'] = np.nan
    df.loc[rng.choice(n, 8, replace=False), 'nota_final'] = np.nan
    outl = rng.choice(n, 4, replace=False)
    df.loc[outl, 'horas_estudo'] = df.loc[outl, 'horas_estudo'].fillna(7) * 10   # vírgula esquecida
    df = pd.concat([df, df.sample(12, random_state=seed)], ignore_index=True)   # duplicados
    return df.sample(frac=1, random_state=seed).reset_index(drop=True)

gerar_dados().to_csv('alunos.csv', index=False)
print('alunos.csv criado!')

## 1 · Inspecionar

Carregue o arquivo e olhe as primeiras linhas.

In [ ]:
# TODO: leia o arquivo 'alunos.csv' com o pandas
df = ___
df.head()

In [ ]:
# TODO: quantas linhas e colunas? E quais os tipos de cada coluna?
print(df.___)
df.___()

In [ ]:
# TODO: estatísticas das colunas numéricas. Repare no máximo de horas_estudo!
df.___()

In [ ]:
# TODO: quantos valores faltantes há em cada coluna?
df.___().sum()

In [ ]:
# TODO: quais valores aparecem na coluna 'turno'?
df['turno'].___()

**Para pensar:** liste os problemas que você encontrou. Quais colunas têm faltantes? O que há de estranho em `turno` e em `horas_estudo`?

*Sua resposta:* …

## 2 · Limpar

### 2.1 Duplicados

In [ ]:
# TODO: conte as linhas duplicadas e remova-as
print('duplicados:', df.duplicated().___())
df = df.___()
print('linhas agora:', len(df))

### 2.2 Categorias inconsistentes
Para o computador, `"manhã"`, `"Manhã"` e `"MANHÃ "` são três valores diferentes.
Padronize: tire os espaços das pontas, deixe tudo minúsculo e corrija `"manha"` sem acento.

In [ ]:
# TODO: padronize a coluna turno
df['turno'] = df['turno'].str.___().str.___()
df['turno'] = df['turno'].replace({'manha': 'manhã'})
df['turno'].value_counts()

### 2.3 Outliers
Ninguém estuda 70 horas *extras* por semana: provavelmente alguém esqueceu a vírgula (7,0 → 70).
Há duas saídas: **corrigir** (dividir por 10, como no slide) ou **descartar o valor** (virar `NaN`).
Como **não temos certeza** do valor certo, aqui vamos transformar em `NaN`; depois ele será preenchido como os outros faltantes.

In [ ]:
plt.boxplot(df['horas_estudo'].dropna())
plt.title('horas_estudo antes da limpeza'); plt.show()

# TODO: transforme em NaN os valores de horas_estudo acima de 24
df.loc[df['horas_estudo'] > ___, 'horas_estudo'] = np.nan
df['horas_estudo'].describe()

### 2.4 Alvo faltante
Uma linha sem `nota_final` não ensina nada: remova-a.
Os faltantes nas **features** (horas, faltas) **não** serão preenchidos agora — isso será feito dentro do `Pipeline`, usando só o treino (etapa 4).

In [ ]:
# TODO: remova as linhas sem nota_final
df = df.dropna(subset=[___])
print(len(df), 'linhas')
df.isna().sum()

## 3 · Explorar

In [ ]:
# TODO: histogramas de todas as colunas numéricas
df.___(bins=20, figsize=(12, 6))
plt.tight_layout(); plt.show()

**Média × mediana:** compare as duas para `horas_estudo` e para `renda_familiar`. Em qual delas a diferença é maior? Por quê?

In [ ]:
for col in ['horas_estudo', 'renda_familiar']:
    # TODO: calcule a média e a mediana da coluna
    print(f"{col}: média = {df[col].___():.1f} | mediana = {df[col].___():.1f}")

In [ ]:
# TODO: gráfico de dispersão horas_estudo × nota_final
plt.scatter(df[___], df[___], alpha=.5)
plt.xlabel('horas de estudo'); plt.ylabel('nota final'); plt.show()

In [ ]:
# TODO: correlação entre as colunas numéricas
df.___(numeric_only=True).round(2)

In [ ]:
# TODO: nota média por turno e por monitoria
df.groupby(___)['nota_final'].mean().round(2)

**Para pensar:** quais features parecem ajudar a prever a nota? Alguma parece não ter relação nenhuma?
Lembre: correlação não é causalidade.

*Sua resposta:* …

## 4 · Preparar sem vazamento
### 4.1 Features × alvo — e a coluna que “conta a resposta”
O alvo é `nota_final`. A coluna `situacao` (aprovado/reprovado) é **calculada a partir da nota**: ela só existe *depois* que a nota sai.
Usá-la como feature seria **vazamento de dados** — o modelo “colaria” a resposta. Veja como ela entrega a nota:

In [ ]:
# TODO: menor e maior nota em cada situação
df.groupby('situacao')['nota_final'].agg([___, ___])

In [ ]:
alvo = 'nota_final'
# TODO: escolha as features (todas menos o alvo e a coluna que vaza)
X = df.drop(columns=[alvo, ___])
y = df[alvo]

num = ['horas_estudo', 'faltas', 'idade', 'renda_familiar']
cat = ['turno', 'monitoria']
X.head()

### 4.2 Separar treino e teste **antes** de qualquer transformação
Aqui fazemos só treino (80%) e teste (20%). A “validação” do slide será feita com **validação cruzada** dentro do treino, a partir do Lab 01.

In [ ]:
# TODO: 80% treino, 20% teste, random_state=42
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=___, random_state=42)
print(X_tr.shape, X_te.shape)

### 4.3 O preparo em um `Pipeline`
Cada peça tem um papel:
- `SimpleImputer(strategy='median')` — preenche faltantes com a **mediana** (robusta a outliers, como vimos na EDA);
- `StandardScaler()` — padroniza: `z = (x − μ) / σ`;
- `OneHotEncoder(handle_unknown='ignore')` — uma coluna 0/1 por categoria; se o teste trouxer uma categoria que o treino nunca viu, ela vira só zeros em vez de dar erro;
- `ColumnTransformer` — aplica um tratamento diferente para cada **tipo** de coluna;
- `Pipeline` — encadeia os passos e garante a ordem: tudo é **ajustado (`fit`) só no treino**.

In [ ]:
preparo = ColumnTransformer([
    # TODO: complete com SimpleImputer(strategy='median') e StandardScaler()
    ('num', Pipeline([('imputar', ___), ('padronizar', ___)]), num),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat),
])

# TODO: ajuste no treino e transforme o treino; depois só TRANSFORME o teste
Z_tr = preparo.___(X_tr)
Z_te = preparo.___(X_te)

colunas = preparo.get_feature_names_out()
pd.DataFrame(Z_tr, columns=colunas).head()

### 4.4 O que o `fit` aprendeu — e de onde
O `StandardScaler` guarda a média ($\mu$) e o desvio ($\sigma$) que aprendeu no `fit`.
Ajustado **só no treino**, as colunas do treino ficam com média 0 e desvio 1; as do teste ficam **perto**, mas não exatamente — afinal, o teste é “novo”.

In [ ]:
n_num = len(num)
pd.DataFrame({
    'média treino':  Z_tr[:, :n_num].mean(axis=0),
    'desvio treino': Z_tr[:, :n_num].std(axis=0),
    'média teste':   Z_te[:, :n_num].mean(axis=0),
}, index=num).round(3)

Agora o jeito **errado**: ajustar o scaler com o dataset inteiro (treino + teste) antes de separar.
O $\mu$ aprendido muda — ou seja, **informação do teste entrou no ajuste**. Com poucos dados ou estatísticas mais sensíveis, isso infla o resultado.

In [ ]:
mu_certo = preparo.named_transformers_['num'].named_steps['padronizar'].mean_
# TODO: ajuste um StandardScaler com TODO o X numérico (o jeito errado) e pegue as médias
mu_errado = StandardScaler().___(X[num].fillna(X[num].median())).mean_

pd.DataFrame({'μ só treino (certo)': mu_certo, 'μ treino + teste (errado)': mu_errado}, index=num).round(2)

### 4.5 Salvar os dados limpos
O Lab 01 começa exatamente daqui, com esta versão limpa.

In [ ]:
df.to_csv('alunos_limpo.csv', index=False)
print(df.shape, '→ alunos_limpo.csv')

## 5 · Desafios extras (se sobrar tempo)
### 5.1 Quem é mais parecido com você? (escala em código)
Compare o aluno mais parecido com o primeiro aluno do treino usando a **distância euclidiana**,
com e sem padronização. Use só as colunas numéricas sem faltantes.

In [ ]:
cols = ['faltas', 'idade', 'renda_familiar']
A = X_tr[cols].dropna()
voce, outros = A.iloc[0], A.iloc[1:]

def mais_parecido(dados, ref):
    dist = np.sqrt(((dados - ref) ** 2).sum(axis=1))
    return dist.idxmin()

# TODO: padronize com a média e o desvio de A
Az = (A - A.___()) / A.___()

print('você:', voce.to_dict())
print('sem padronizar →', A.loc[mais_parecido(outros, voce)].to_dict())
print('padronizando   →', A.loc[mais_parecido(Az.iloc[1:], Az.iloc[0])].to_dict())

**Para pensar:** sem padronizar, qual coluna decidiu quem é “parecido”? Por quê?

### 5.2 Label encoding × one-hot
Transforme `turno` em números das duas formas e compare.

In [ ]:
label = df['turno'].map({'noite': 0, 'manhã': 1, 'tarde': 2})
# TODO: one-hot da coluna turno com o pandas
onehot = pd.___(df['turno'], prefix='turno', dtype=int)
pd.concat([df['turno'], label.rename('label'), onehot], axis=1).head()

**Para pensar:** no label encoding, “tarde” (2) é o dobro de “manhã” (1)? Que problema isso causa num modelo que faz contas com esses números?

## Para discutir
1. Transformar os outliers em `NaN` ou dividir por 10: qual decisão você tomaria? O que muda nos dados?
2. Por que preencher os faltantes **dentro** do pipeline, e não antes do `train_test_split`?
3. Que outra coluna, num problema real, poderia “contar a resposta” como a `situacao`?